# Symbolic Regression using Sparse Identification of Nonlinear Dynamics (SINDy)

In [ ]:
# Load libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy.integrate import solve_ivp

# Clone the repository to be able to access custom utilities
# for this tutorial. This is only needed if you are running 
# this notebook on Google Colab.
import os
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    if not os.path.exists('I-X_technical_tutorial_sindy'):
        !git clone https://github.com/stroblmar/I-X_technical_tutorial_sindy.git
    %pip install pysindy
    %cd I-X_technical_tutorial_sindy

# Load custom utilities
from utils import LV_PALETTE, generate_lv_data

# Introduction to Model Discovery and Symbolic Regression

### What is model discovery?
Model discovery is the process of learning the governing equations of a dynamical system from data. We typically have state variables (e.g. x, y, z) that evolve over time (and potentially space, though we will not cover that here) according to some unknown governing equations, given by:
\begin{align}
\frac{dx}{dt} = f(x,y,z) \\
\frac{dy}{dt} = g(x,y,z) \\
\frac{dz}{dt} = h(x,y,z)
\end{align}

and we seek to learn either the model structure (i.e. the functional forms of f, g, h), and/or the associated parameters (e.g. coefficients of the functional forms inside f, g, h), or both.

Examples: 
  - Physics: weather prediction, fluid dynamics (e.g. building airplane models).
  - Biology: epidemiology (e.g. SIR models during COVID-19), population dynamics (e.g. ecosystem management, cancer modelling), gene regulatory networks.

Taditionally done through a loop in which a model is proposed, tested against data, and refined until it fits the data well enough. This is a time-consuming process that requires a lot of domain knowledge and expertise.

Side note: the process of "discarding" a model is in itself a key part of the scientific method (potentially arguable THE key part). Scientists can never prove something. They can only fail to disprove it. A model that fails tells us a key assumption of our model is wrong or missing!

![alt text](figures/figures.002.png)

### Symbolic Regression
However, this same process is also **quite time-consuming** and requires a lot of domain knowledge and expertise. With that domain "knowledge" also comes **bias**: We may only propose models that we are familiar with and have seen before. 

Symbolic regression seeks to overcome these limitations. It is a family of methods which seek to automate the discovery process in order to make it more efficient and less prone to human bias.

Broadly, there are three main approaches to symbolic regression:
1. **Genetic programming (GP):** a method that uses evolutionary algorithms to evolve mathematical expressions that fit the data. For example, PySR is a Python library that implements GP for symbolic regression.
2. **Sparse Identification of Nonlinear Dynamics (SINDy):** a method that uses sparse regression on a candidate library of terms to identify the right-hand side of the differential equation model. This is what we will cover today.
3. **Symbolic Transformers:** transformer-based embedding models trained to map ODE trajectories to equations. See [Farbod's tutorial](https://colab.research.google.com/github/farbodf91/ix-verifier-guided-discovery/blob/main/01_IX_symbolic_discovery_tutorial.ipynb) for more details 

## Sparse Identification of Nonlinear Dynamics (SINDy)


Sparse Identification of Nonlinear Dynamics (SINDy) is a method for dynamics discovery which was first proposed by Brunton et al in 2016 ([Brunton et al (2016)](https://www.pnas.org/doi/10.1073/pnas.1517384113)). It is a very flexible and scalable approach and computationally inexpensive than other alternatives. As a result, it has rapidly been adopted in a number of fields ranging from [plasma physics](https://arxiv.org/abs/2004.10389) to [cancer research](https://doi.org/10.3389/fimmu.2023.1115536) and many additional extensions have been developed (e.g. [Bayesian SINDy](https://github.com/llfung/ODR-BINDy) (by our own Lloyd Fung!), [Ensemble SINDy](https://royalsocietypublishing.org/rspa/article/478/2260/20210904/56618/Ensemble-SINDy-Robust-sparse-model-discovery-in), [SINDy for PDEs](https://pubmed.ncbi.nlm.nih.gov/28508044/)).

In this tutorial, I will explain the basic SINDy algorithm and demonstrate how to use it to discover the governing equations of a simple dynamical system. We will also explore some of the key hyperparameters that can be tuned to improve the performance of the algorithm. For more details and many more (great!) examples, I recommend the [pysindy docs](https://pysindy.readthedocs.io/en/latest/index.html) and the resources from the Symbolic Model Discovery Workshop organised at I-X in 2025 (see [here](https://www.symbolicmodel.org/tutorial-and-recording)).

### The principle intuition behind SINDy

The key observation underlying SINDy is that the right-hand-sides of many dynamical systems are linear in their coefficients. This means that while the RHS may involve terms which are nonlinear in the state variables, such as $x^2$ or $x_1 x_2$, these nonlinear terms combine in a linear way to give the overall dynamics, where the coefficients/parameters are the weights in this weighted sum. For example, take the Lorenz equations. We can re-write them as follows:

\begin{equation*}
    \dot{\mathbf{x}} 
    =
    \begin{bmatrix} \dot{x} \\ \dot{y} \\ \dot{z} \end{bmatrix}
    =
    \begin{bmatrix}
        \sigma(y-x) \\
        x(\rho-z)-y \\
        xy - \beta z
    \end{bmatrix}
    =
    \begin{bmatrix}
        (-\sigma) x + \sigma y \\
        \rho x + (-1) y + (-1) xz \\
        (-\beta) z + xy
    \end{bmatrix}
    =
    \begin{bmatrix}
        -\sigma & \sigma & 0      & 0  & 0 \\
        \rho    & -1     & 0      & -1 & 0 \\
        0       & 0      & -\beta & 0  & 1
    \end{bmatrix}
    \begin{bmatrix} x \\ y \\ z \\ xz \\ xy \end{bmatrix}
\end{equation*}

where we have decomposed the RHS into a product of a vector which depends solely on the state variables (in potentially non-linear ways), and a matrix of coefficients, which specify how those terms are combined to form each equation. 

This observation is important because it allows us to recast the problem of finding the governing equations as a linear regression problem. Suppose we propose a collection of candidate functional terms that might appear on the RHS of the dynamical system we wish to identify, and collate them into a *function library*:

\begin{equation}
    \boldsymbol{\theta}(\mathbf{x}) = \begin{bmatrix} 1 & x & y & z & xy & xz & yz & x^2 & \cdots \end{bmatrix}.
\end{equation}

Crucially, we can calculate each term in $\boldsymbol{\theta}(\mathbf{x})$ directly from our observations of the  system. 

The problem of discovering the governing equations then reduces to finding the coefficient matrix $\boldsymbol{\Xi}$ such that

\begin{equation}
    \dot{\mathbf{x}} = \boldsymbol{\theta}(\mathbf{x})\,\boldsymbol{\Xi}.
\end{equation}

In other words, we are solving a linear regression problem in which the derivatives become the "response" variable, which we seek to predict using our function library $\boldsymbol{\theta}(\mathbf{x})$ as the "explanatory variables".

### The SINDy algorithm

Let us assume we are trying to learn the governing equations of a system with $n$ state variables, $\mathbf{x} = [x_1, x_2, \ldots, x_n]^\top$. To do so, we have collected $m$ observations of the system at different points in time, which we will denote by $\mathbf{\hat{x}}_i = \mathbf{\hat{x}}(t_i) = [x_1(t_i), x_2(t_i), \ldots, x_n(t_i)]^\top$, for $i = 1, 2, \ldots, m$. We can assemble these observations into a data matrix 

\begin{equation}
    \mathbf{\hat{X}} =
    \;=\;
    \begin{bmatrix}
        \hat{x}_1(t_1) & \hat{x}_2(t_1) & \cdots & \hat{x}_n(t_1) \\
        \hat{x}_1(t_2) & \hat{x}_2(t_2) & \cdots & \hat{x}_n(t_2) \\
        \vdots    & \vdots    & \ddots & \vdots \\
        \hat{x}_1(t_m) & \hat{x}_2(t_m) & \cdots & \hat{x}_n(t_m)
    \end{bmatrix}.
\end{equation}

Note, that here and throughout, $x_i(t_j)$ denotes the value of the $i$-th state variable at time $t_j$ (the ground truth), and $\hat{x}_i(t_j)$ denotes our observed value of that state variable at that time. Ideally, of course, the two are the same, but in practice measurement noise and other sources of error typically mean that they are not.

Applying SINDy to this data proceeds in five main steps:

#### Step 1: Obtain observations of the time derivatives of the state variables
In the first step, we use the data matrx to estimate the time derivatives of the state variables, $\dot{\mathbf{x}} = [\dot{x}_1, \dot{x}_2, \ldots, \dot{x}_n]^\top$, at each observation time. We can then assemble these estimated derivatives into a second matrix with the observed derivatives,

\begin{equation}
    \dot{\mathbf{\hat{X}}} =
    \begin{bmatrix}
        \dot{\hat{x}}_1(t_1) & \dot{\hat{x}}_2(t_1) & \cdots & \dot{\hat{x}}_n(t_1) \\
        \dot{\hat{x}}_1(t_2) & \dot{\hat{x}}_2(t_2) & \cdots & \dot{\hat{x}}_n(t_2) \\
        \vdots          & \vdots          & \ddots & \vdots \\
        \dot{\hat{x}}_1(t_m) & \dot{\hat{x}}_2(t_m) & \cdots & \dot{\hat{x}}_n(t_m)
    \end{bmatrix},
\end{equation}

where $\dot{x_i}(t)$ denotes the time derivative of $x_i$ evaluated at time $t$, $\frac{dx_i}{dt}(t)$, and $\dot{\hat{x}}_i(t)$ denotes our estimate of that derivative at that time. 

To obtain these estimates, we typically use finite difference methods. Crucially, as for any statistical method the output of SINDy is only as good as the input data. If the derivative estimates are poor, the resulting model will be poor as well. In this particular context, derivative estimates are particularly sensitive to noise in the data, and so it is important to use a method that is robust to noise, and potentially apply appropriate smoothing or filtering to the data before estimating derivatives.

#### Step 2: Propose a library of candidate functions
In the second step, we propose a library of candidate functions that might appear on the right-hand side of the governing equations. This library is typically constructed from a set of basis functions, such as polynomials, trigonometric functions, or other functions that are thought to be relevant to the system. The library is then evaluated at each observation time, and the resulting values are assembled into a third matrix, which we call the *library matrix*, 
\begin{equation}
    \mathbf{\Theta}(\mathbf{\hat{X}}) =
    \begin{bmatrix}
        \theta_1(\mathbf{\hat{x}}(t_1)) & \theta_2(\mathbf{\hat{x}}(t_1)) & \cdots & \theta_p(\mathbf{\hat{x}}(t_1)) \\
        \theta_1(\mathbf{\hat{x}}(t_2)) & \theta_2(\mathbf{\hat{x}}(t_2)) & \cdots & \theta_p(\mathbf{\hat{x}}(t_2)) \\
        \vdots          & \vdots          & \ddots & \vdots \\
        \theta_1(\mathbf{\hat{x}}(t_m)) & \theta_2(\mathbf{\hat{x}}(t_m)) & \cdots & \theta_p(\mathbf{\hat{x}}(t_m))
    \end{bmatrix},
\end{equation}

where $\theta_j(\mathbf{x})$ denotes the $j$-th candidate function evaluated at the state vector $\mathbf{x}$, and $p$ is the total number of candidate functions in the library.

For example, if we propose a library of polynomial functions up to degree 2, the library matrix would be
\begin{equation}
    \mathbf{\Theta}(\mathbf{\hat{X}}) =
    \begin{bmatrix}
        1 & \hat{x}_1(t_1) & \hat{x}_2(t_1) & \cdots & \hat{x}_n(t_1) & \hat{x}_1(t_1)^2 & \hat{x}_1(t_1)\hat{x}_2(t_1) & \cdots & \hat{x}_n(t_1)^2 \\
        1 & \hat{x}_1(t_2) & \hat{x}_2(t_2) & \cdots & \hat{x}_n(t_2) & \hat{x}_1(t_2)^2 & \hat{x}_1(t_2)\hat{x}_2(t_2) & \cdots & \hat{x}_n(t_2)^2 \\
        \vdots          & \vdots          & \vdots          & \ddots & \vdots          & \vdots          & \vdots          & \ddots & \vdots          \\
        1 & \hat{x}_1(t_m) & \hat{x}_2(t_m) & \cdots & \hat{x}_n(t_m) & \hat{x}_1(t_m)^2 & \hat{x}_1(t_m)\hat{x}_2(t_m) & \cdots & \hat{x}_n(t_m)^2
    \end{bmatrix}.
\end{equation}

Choosing the right library of candidate functions is crucial to the success of SINDy. If the true governing equations contain terms that are not included in the library, SINDy will not be able to identify them. Conversely, if the library contains too many irrelevant terms, SINDy may overfit the data and produce a model that does not generalize well to new data. Importantly, what constitutes a "success" and a "good" library also depends on what the goal of the analysis is. Is our goal to build a highly accurate model, and perhaps one that runs fast and efficiently? In that case we might not care about whether the true terms are all contained and instead might satisfy efficient approximations. Conversely, if we have spefic library terms in mind - for example, from prior knowledge or subject expertise - and we want to know what the best approximation of the system is using just those terms, we might choose a smaller, curated library, at the expense of model accuracy.

#### Step 3: Solve the sparse regression problem
We are now ready to perform inference. To do so, we solve the following multi-variate, multiple, linear regression problem
\begin{equation}
    \dot{\mathbf{\hat{X}}} = \mathbf{\Theta}(\mathbf{\hat{X}})\,\boldsymbol{\Xi},
\end{equation}
where $\boldsymbol{\Xi}$ is the coefficient matrix we wish to estimate, and has the form
\begin{equation}
    \boldsymbol{\Xi} =
    \begin{bmatrix}
        \xi_{11} & \xi_{12} & \cdots & \xi_{1n} \\
        \xi_{21} & \xi_{22} & \cdots & \xi_{2n} \\
        \vdots   & \vdots   & \ddots & \vdots   \\
        \xi_{p1} & \xi_{p2} & \cdots & \xi_{pn}
    \end{bmatrix},
\end{equation}
where $\xi_{ij}$ is the coefficient of the $i$-th library term in the equation for the $j$-th state variable. This means that the estimated governing equation for $j$-th state variable is given by
\begin{equation}
    \dot{x}_j = \xi_{1j} \theta_1(\mathbf{x}) + \xi_{2j} \theta_2(\mathbf{x}) + \cdots + \xi_{pj} \theta_p(\mathbf{x}).
\end{equation}

To solve this problem, we can use any standard linear regression method, including standard ordinary least squares (OLS). 

#### Step 4: Encourage sparsity in the coefficient matrix
While Step 3 will give us a solution for the coefficient matrix $\boldsymbol{\Xi}$, it is likely to be dense (i.e., many of the coefficients will be non-zero). However, we expect that only a few of the candidate functions will be relevant to the governing equations, and so we want to encourage sparsity in the coefficient matrix.

There are different ways to do so, including using a sparse regression method, such as LASSO or sequential thresholding. These methods add a penalty term to the regression objective function that encourages small coefficients to be set to zero, effectively performing feature selection and yielding a sparse solution. Here we will use sequential thresholding, which is a simple and effective method for encouraging sparsity in the coefficient matrix, and is the method [Brunton et al (2016)](https://www.pnas.org/doi/10.1073/pnas.1517384113) used in their original work. 

The idea of the Sequentially Thresholded Least Squares (STLS) algorithm is to iteratively fit the regression model, and at each iteration, set any coefficients that are below a certain threshold, $\lambda$, to zero. This process is repeated until convergence, yielding a sparse solution for the coefficient matrix.

#### Step 5: Model validation and selection
Finally - and crucially - once we have built a model, the true art and science of modelling begins. We assess the model's performance on tasks such as prediction, interpretation, and generalization - depending on our use case. We may compare the model's performance against alternative models, and select the best one based on some criteria, such as AIC, BIC, or cross-validation error. Importantly, depending on whether the goal is to build a highly accurate model, or to identify the true governing equations, the goal here the key information is in either the best fitting model - or sometimes it may be in the models which we discarded.

### Example: SINDy on the Lotka-Volterra equations

To illustrate how SINDy works in practice, we will apply it to some simulated data where we know the underlying governing equations. In this way, we can assess how well SINDy is able to recover the true governing equations from the data.

My research focuses on the ecology between different populations of cancer cells. So, I have chosen to illustrate the use of SINDy on the Lotka-Volterra model of competition. This model is one of the most well-known models in ecology and describes the population dynamics between two species in an ecosystem. It has been used extensively to understand under what conditions species can co-exist and what determines the stability of ecosystems. 

Let $x(t)$ denote the density of Species X, and $y(t)$ denote the density of Species Y. The Lotka-Volterra equations are given by
\begin{align}
    \dot{x} &= \underbrace{r_x x}_{\text{growth}} - \underbrace{c_{xx} x x}_{\text{intra-specific competition between members of X}} - \underbrace{c_{yx} x y}_{\text{inter-specific competition between X and Y}} \\
    \dot{y} &= \underbrace{r_y y}_{\text{growth}} - \underbrace{c_{yy} y y}_{\text{intra-specific competition between members of Y}} - \underbrace{c_{xy} y x}_{\text{inter-specific competition between Y and X}}
\end{align}

where $r_x$ and $r_y$ are the intrinsic growth rates of Species X and Y, respectively, and $c_{xx}$, $c_{yx}$, etc are the competition coefficients that quantify the effect of interactions on growth (e.g. the effect of Species Y on Species X is given by $c_{yx}$). We assume that we start with a total initial population of size $n_0 = x(0) + y(0)$, where the initial fraction of Species Y is given by $f_y = y(0)/n_0$.

#### 1) The Data
For this example, we will use the following parameter values:
- $r_x = 0.1$
- $r_y = 0.08$
- $c_{xx} = r_x$
- $c_{yy} = r_y$
- $c_{yx} = 0.05$
- $c_{xy} = 0.02$
- $n_0 = 0.1$
- $f_y = 0.5$

and we will assume that we have collected data on each population once a day for 100 days, and that for simplicity we have no measurement noise. 

In [ ]:
# Define the parameters of the Lotka-Volterra model
params_dict = {
    "r_x": 0.1,  # growth rate of species X
    "r_y": 0.08,  # growth rate of species Y
    "c_xx": 0.1,  # intra-specific competition coefficient for species X
    "c_yy": 0.08,  # intra-specific competition coefficient for species Y
    "c_yx": 0.05,  # inter-specific competition coefficient from species Y to X
    "c_xy": 0.02,  # inter-specific competition coefficient from species X to Y
}
ic_dict = {
    "n0": 0.1,  # total initial population size
    "f_y": 0.5,  # initial fraction of species Y
}
duration_of_experiment = 100  # duration of observation in time units
sampling_interval = 1 # interval at which we are collecting observations

# Simulate the Lotka-Volterra dynamics and generate observations. To facilitate this, I have written 
# a function that will do this for us. It takes in the parameters and details of the experiment we 
# want to run and returns a pandas DataFrame containing the observations of the two species over time.
observations_df = generate_lv_data(params_dict=params_dict, ic_dict=ic_dict,
                 sampling_interval=sampling_interval, t_end=duration_of_experiment)

# Inspect observations
observations_df.head(10)

In [ ]:
# Plot the dynamics of the two species over time
plt.figure(figsize=(8, 4))
sns.scatterplot(data=observations_df, x='t', y='x', label='Species X', 
                color=LV_PALETTE['x'], s=50, marker='o',
                edgecolor='black', linewidth=1, zorder=10)
sns.scatterplot(data=observations_df, x='t', y='y', label='Species Y', 
                color=LV_PALETTE['y'], s=50, marker='s',
                edgecolor='black', linewidth=1, zorder=10)
# sns.lineplot(data=observations_df, x='t', y='x', label='Species X', 
#              color='blue', linewidth=2, zorder=5)
# sns.lineplot(data=observations_df, x='t', y='y', label='Species Y', 
#              color='orange', linewidth=2, zorder=5)
plt.title('Observed Data')
plt.xlabel('Time')

#### 2) Setting up the SINDy problem

To apply SINDy, we need to define the following three components:
1. The **derivative matrix**, $\dot{\mathbf{\hat{X}}}$, which contains the estimated time derivatives of the state variables at each observation time.
2. The **library matrix**, $\mathbf{\Theta}(\mathbf{\hat{X}})$, which contains the evaluated candidate functions at each observation time.
3. The **sparsity threshold**, $\lambda$, in the STLS algorithm, which determines how strict we are in enforcing sparsity in the coefficient matrix.

Let's first obtain the derivatives:

In [ ]:
# ------------------------------------------------------------------
# 1. Estimating the derivative matrix, $\dot{\mathbf{\hat{X}}}$
# ------------------------------------------------------------------
# For convenience, we will use the `np.gradient()` function from numpy which implements a 2nd-order 
# central difference method to compute the numerical derivative of a time series. Note that the accuracy of 
# the method we use will directly affect the accuracy of the SINDy model we learn!
X_dot_obs = np.gradient(observations_df[['x', 'y']], sampling_interval, axis=0)

# Let's look at the first 10 rows of the derivative matrix to see what it looks like:
X_dot_obs[:10, :]

Next, we construct the library matrix. For this example, we will use a library of polynomial functions. That's a reasonable starting point here because many well-known models in ecology use polynomial terms to describe the interactions between species. In practice, choosing the right library is a key part of the SINDy workflow, and requires careful thought. The choice of library will depend on the system being studied and the goals of the analysis. 

In [ ]:
# ------------------------------------------------------------------
# 2. Building the library matrix, $\mathbf{\Theta}(\mathbf{\hat{X}})$
# ------------------------------------------------------------------

n_observations = observations_df.shape[0]

# Add a column for the constant term (1)
theta = np.ones((n_observations, 1))

# Let's also create a list of the names of the library functions we have included to help us keep track of them:
library_function_names = ['1']

# Inspect the first 10 rows of the library matrix to see what it looks like:
theta[:10, ]

In [ ]:
# Next, let's add the linear terms for each species (i.e. x and y):
theta = np.column_stack((
    theta,
    observations_df[['x', 'y']].values
))
library_function_names.extend(['x', 'y'])

theta[:10, ]

In [ ]:
# Next, let's add the quadratic terms
theta = np.column_stack((
    theta,
    observations_df[['x', 'y']].values ** 2 # the x^2 and y^2 terms
))
theta = np.column_stack((
    theta,
    observations_df['x'].values * observations_df['y'].values  # the x y term
))
library_function_names.extend(['x^2', 'y^2', 'x y'])

# Inspect
theta[:10, ]

In [ ]:
# Let's print the final shape of the library matrix and the names of the functions we have included in it:
print(f"Number of library functions: {theta.shape[1]}")
print(f"Library function names: {library_function_names}")
print(f"Number of observations: {theta.shape[0]}")
print(f"Final library matrix shape: {theta.shape}")

We are now ready to perform inference. To do so, we solve the following multi-variate, multiple, linear regression problem
\begin{equation}
    \dot{\mathbf{\hat{X}}} = \mathbf{\Theta}(\mathbf{\hat{X}})\,\boldsymbol{\Xi},
\end{equation}

for the coefficient matrix $\boldsymbol{\Xi}$, whose entries determine which candidate functions in the library contribute to each governing equation, and with what weight.

The key advantage of this formulation is that it reduces a problem of dynamical systems identification to one of linear regression — a well-studied problem for which robust, scalable solvers already exist. Let's ignore the need for sparsity for now and just use ordinary least squares (OLS) to solve the regression problem. We can do so in a one-liner:

In [ ]:
# Solve for the coefficient matrix $\boldsymbol{\Xi}$ using ordinary least squares (OLS):
ols_results = np.linalg.lstsq(theta, X_dot_obs, rcond=None)

# Unpack the results of the OLS regression:
xi = ols_results[0] # The least-squares solution for the coefficient matrix
residuals = ols_results[1] # The residuals of the regression

print(f"Mean squared error of the regression: {residuals / n_observations}")
xi

Let's test how well the obtained solution matches the observed data:

In [ ]:
def sindy_rhs(t, x, xi):
    """
    Compute the right-hand side of the SINDy model in a way that's compatible with scipy's ODE solvers.
    
    Parameters:
    - t: time (not used in this function, but included for compatibility with ODE solvers)
    - x: state vector (current values of the species)
    - xi: coefficient matrix (learned from regression)
    
    Returns:
    - dxdt: time derivative of the state vector
    """
    # Evaluate the library functions at the current state
    # theta_eval = np.array([1, x[0], x[1], x[0]**2, x[1]**2, x[0]*x[1], 
    #                        x[0]**3, x[1]**3, x[0]**2*x[1], x[0]*x[1]**2])
    theta_eval = np.array([1, x[0], x[1], x[0]**2, x[1]**2, x[0]*x[1]])
    
    # Compute the time derivative using the learned coefficients
    dxdt = theta_eval @ xi
    
    return dxdt

# Solve the SINDy model using the learned coefficients and compare it to the original Lotka-Volterra dynamics
t_span = (0, duration_of_experiment)
t_eval = observations_df['t'].values
x0 = observations_df[['x', 'y']].iloc[0].values  # initial conditions
solution = solve_ivp(
    fun=sindy_rhs,
    t_span=t_span,
    y0=x0,
    args=(xi,),
    t_eval=t_eval,
    method='LSODA',
    rtol=1e-12,
    atol=1e-12,
    max_step=0.1
)

# Re-format into a DataFrame for plotting
prediction_df_sindy = pd.DataFrame(solution.y.T, columns=['x', 'y'], index=t_eval)
prediction_df_sindy.reset_index(inplace=True)
prediction_df_sindy.rename(columns={'index': 't'}, inplace=True)
prediction_df_sindy.head()

In [ ]:
# Plot the predicted dynamics on top of the observed data
plt.figure(figsize=(12, 6))
sns.scatterplot(data=observations_df, x='t', y='x', label='Species X', 
                color=LV_PALETTE['x'], s=50, marker='o',
                edgecolor='black', linewidth=1, zorder=5)
sns.scatterplot(data=observations_df, x='t', y='y', label='Species Y', 
                color=LV_PALETTE['y'], s=50, marker='s',
                edgecolor='black', linewidth=1, zorder=5)
sns.lineplot(data=prediction_df_sindy, x='t', y='x', label='SINDy Prediction for X', 
             color=LV_PALETTE['x_sindy'], linewidth=2, zorder=10)
sns.lineplot(data=prediction_df_sindy, x='t', y='y', label='SINDy Prediction for Y', 
             color=LV_PALETTE['y_sindy'], linewidth=2, zorder=10)
plt.ylim(0, 1)
plt.title('Observed Data vs SINDy Predictions')
plt.xlabel('Time')
plt.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', ncol=4, borderaxespad=0)
plt.tight_layout()

The model fit is excellent! How about the inferred model structure? Let's take a look at the coefficient matrix:

In [ ]:
# Turn the coefficient matrix into a pandas DataFrame for easier interpretation
xi_df = pd.DataFrame(xi, index=library_function_names, columns=['x_dot', 'y_dot'])
xi_df

In [ ]:
# To make it easy to assess the accuracy of the learned coefficients, let's 
# annotate the dataframe with the ground truth parameters. For reference, here are the equations again:
# dx/dt = r_x * x - c_xx * x^2 - c_yx * x * y
# dy/dt = r_y * y - c_yy * y^2 - c_xy * x * y
# Library order: ['1', 'x', 'y', 'x^2', 'y^2', 'x y', 'x^3', 'y^3', 'x^2 y', 'x y^2']
x_dot_true = [0, params_dict['r_x'], 0, -params_dict['c_xx'], 0, -params_dict['c_yx']]
y_dot_true = [0, 0, params_dict['r_y'], 0, -params_dict['c_yy'], -params_dict['c_xy']]
xi_df['x_dot_true'] = x_dot_true
xi_df['y_dot_true'] = y_dot_true

# Compute the relative error between the learned coefficients and the true coefficients
xi_df['x_dot_error'] = np.abs(xi_df['x_dot'] - xi_df['x_dot_true']) / (np.abs(xi_df['x_dot_true']) + 1e-12)
xi_df['y_dot_error'] = np.abs(xi_df['y_dot'] - xi_df['y_dot_true']) / (np.abs(xi_df['y_dot_true']) + 1e-12)
xi_df['x_dot_error'] = xi_df['x_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['y_dot_error'] = xi_df['y_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['x_dot_error'] = xi_df['x_dot_error'] * 100  # Convert to percentage
xi_df['y_dot_error'] = xi_df['y_dot_error'] * 100  # Convert to percentage

# Make a heatmap to visualise the error
plt.figure(figsize=(10, 6))
sns.heatmap(xi_df[['x_dot_error', 'y_dot_error']],
            annot=True, fmt=".2f", cmap="viridis", cbar_kws={'label': 'Relative Error'},
            xticklabels=['x_dot', 'y_dot'], yticklabels=library_function_names)
plt.title('Relative Error in Learned Coefficients (in %)');

We make two observations:
- For the terms that are part of the true governing equations, the learned coefficients are very close to the true values. This is a good sign, and suggests that we are on the right track in identifying the governing equations.
- However, we also see that the coefficient matrix is dense, with some weight given to all of the proposed candidate functions, including those terms which are not part of either of the true governing equations. This is not too surprising: by virtue of how we constructed the library matrix the columns are highly correlated. In addition, they have very different scales. Both are known issues for OLS regression, and why we need to enforce sparsity in the coefficient matrix. So, let's now move on to the last step and do this.

#### 3) Apply STLSQ to performe sparse regression

Let's apply the STLSQ algorithm to enforce sparsity in the coefficient matrix. We will use a threshold of $\lambda = 0.01$ for now, but we will explore the effect of this threshold on the results later.

In [ ]:
from sklearn.linear_model import ridge_regression

def STLQ(theta, x_dot, threshold, n_iterations=10):
    n_states = x_dot.shape[1]  # Number of state variables (e.g., 2 for Lotka-Volterra)
    # Start by solving the least-squares problem to get an initial guess for the coefficient matrix
    xi = np.linalg.lstsq(theta, x_dot)[0]

    for _ in range(n_iterations): # Sequential thresholding
        small_coeffs = (np.abs(xi) < threshold) # Find small coefficients
        xi[small_coeffs] = 0 # and threshold
        
        for i in range(n_states):  # n_states is state dimension
            big_coeffs_idx = ~small_coeffs[:, i]
            # regress dynamics onto remaining terms to find sparse Xi
            xi[big_coeffs_idx, i] = np.linalg.lstsq(theta[:, big_coeffs_idx], x_dot[:, i])[0]
    return xi
    
# Perform sparse regression
threshold = 0.01 # this is the sparsification parameter: coefficients below this value are set to zero
xi = STLQ(theta, X_dot_obs, threshold, n_iterations=25)
xi


In [ ]:
# Let's examine our new solution
solution = solve_ivp(
    fun=sindy_rhs,
    t_span=t_span,
    y0=x0,
    args=(xi,),
    t_eval=t_eval,
)

# Re-format into a DataFrame for plotting
prediction_df_sindy = pd.DataFrame(solution.y.T, columns=['x', 'y'], index=t_eval)
prediction_df_sindy.reset_index(inplace=True)
prediction_df_sindy.rename(columns={'index': 't'}, inplace=True)

# Plot the predicted dynamics on top of the observed data
plt.figure(figsize=(12, 6))
sns.scatterplot(data=observations_df, x='t', y='x', label='Species X', 
                color=LV_PALETTE['x'], s=50, marker='o',
                edgecolor='black', linewidth=1, zorder=5)
sns.scatterplot(data=observations_df, x='t', y='y', label='Species Y', 
                color=LV_PALETTE['y'], s=50, marker='s',
                edgecolor='black', linewidth=1, zorder=5)
sns.lineplot(data=prediction_df_sindy, x='t', y='x', label='SINDy Prediction for X', 
             color=LV_PALETTE['x_sindy'], linewidth=2, zorder=10)
sns.lineplot(data=prediction_df_sindy, x='t', y='y', label='SINDy Prediction for Y', 
             color=LV_PALETTE['y_sindy'], linewidth=2, zorder=10)
plt.title('Observed Data vs SINDy Predictions')
plt.xlabel('Time')
plt.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', ncol=4, borderaxespad=0)
plt.tight_layout()

In [ ]:
# Let's examine the new coefficient matrix:
xi_df = pd.DataFrame(xi, index=library_function_names, columns=['x_dot', 'y_dot'])
xi_df

In [ ]:
# Compute the relative error between the learned coefficients and the true coefficients
xi_df['x_dot_true'] = x_dot_true
xi_df['y_dot_true'] = y_dot_true
xi_df['x_dot_error'] = np.abs(xi_df['x_dot'] - xi_df['x_dot_true']) / (np.abs(xi_df['x_dot_true']) + 1e-12)
xi_df['y_dot_error'] = np.abs(xi_df['y_dot'] - xi_df['y_dot_true']) / (np.abs(xi_df['y_dot_true']) + 1e-12)
xi_df['x_dot_error'] = xi_df['x_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['y_dot_error'] = xi_df['y_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['x_dot_error'] = xi_df['x_dot_error'] * 100  # Convert to percentage
xi_df['y_dot_error'] = xi_df['y_dot_error'] * 100  # Convert to percentage

# Make a heatmap to visualise the error
plt.figure(figsize=(10, 6))
sns.heatmap(xi_df[['x_dot_error', 'y_dot_error']],
            annot=True, fmt=".2f", cmap="viridis", cbar_kws={'label': 'Relative Error'},
            xticklabels=['x_dot', 'y_dot'], yticklabels=library_function_names)
plt.title('Relative Error in Learned Coefficients (in %)');

In [ ]:
# Let's write out the final equations:
for i, state in enumerate(['x', 'y']):
    equation_terms = []
    for j, func in enumerate(library_function_names):
        coeff = xi[j, i]
        if np.abs(coeff) > 0:
            equation_terms.append(f"{coeff:.5f} * {func}")
    equation_str = " + ".join(equation_terms)
    print(f"d{state}/dt = {equation_str}")

In [ ]:
# In comparison, here are the ground truth equations:
for i, state in enumerate(['x', 'y']):
    equation_terms = []
    for j, func in enumerate(library_function_names):
        coeff = x_dot_true[j] if state == 'x' else y_dot_true[j]
        if np.abs(coeff) > 0:
            equation_terms.append(f"{coeff:.5f} * {func}")
    equation_str = " + ".join(equation_terms)
    print(f"d{state}/dt = {equation_str}")

In short: success!

## pySINDy

The `pysindy` package offers a set of useful tools for applying SINDy to data. It provides a simple interface for defining the library of candidate functions, estimating derivatives, and solving the sparse regression problem. For example, we can perform the same analysis as above using `pysindy` with just a few lines of code:

In [ ]:
import pysindy as ps

# 1) Choose the differentiation method
differentiation_method = ps.FiniteDifference(order=2) # central difference 2nd order

# 2) Choose library
feature_library = ps.PolynomialLibrary(degree=2) # polynomial library up to degree 2

# 3) Select optimiser
optimiser = ps.STLSQ(threshold=0.01, alpha=0) # sequential thresholded least squares

# 4) Instantiate and fit the SINDy model
model = ps.SINDy(optimizer = optimiser, 
                 differentiation_method = differentiation_method,
                 feature_library=feature_library)
model.fit(observations_df[["x", "y"]].values, t=observations_df["t"].values, feature_names=["x", "y"])

# Print the learned model
model.print()

In [ ]:
# Simulate the model
x_predicted_sindy = model.simulate(x0 = x0,  # initial conditions
               t = t_eval # time points at which to evaluate the solution
           )

# Re-format into a DataFrame for plotting
prediction_df_pysindy = pd.DataFrame(x_predicted_sindy, columns=['x', 'y'], index=t_eval)
prediction_df_pysindy.reset_index(inplace=True)
prediction_df_pysindy.rename(columns={'index': 't'}, inplace=True)
prediction_df_pysindy.head()

In [ ]:
# Plot the predicted dynamics on top of the observed data
plt.figure(figsize=(12, 6))
sns.scatterplot(data=observations_df, x='t', y='x', label='Species X', 
                color=LV_PALETTE['x'], s=50, marker='o',
                edgecolor='black', linewidth=1, zorder=5)
sns.scatterplot(data=observations_df, x='t', y='y', label='Species Y', 
                color=LV_PALETTE['y'], s=50, marker='s',
                edgecolor='black', linewidth=1, zorder=5)
sns.lineplot(data=prediction_df_pysindy, x='t', y='x', label='SINDy Prediction for X', 
             color=LV_PALETTE['x_sindy'], linewidth=2, zorder=10)
sns.lineplot(data=prediction_df_pysindy, x='t', y='y', label='SINDy Prediction for Y', 
             color=LV_PALETTE['y_sindy'], linewidth=2, zorder=10)
plt.title('Observed Data vs SINDy Predictions')
plt.xlabel('Time')
plt.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', ncol=4, borderaxespad=0)
plt.tight_layout()

In addition, `pysindy` provides a number of useful features, including:

In [ ]:
# A handy interface for inspecting models
model

In [ ]:
# Similarly for the feature library
feature_library

In [ ]:
# Accessing the feature names
model.get_feature_names()

In [ ]:
# Accessing the learned equations
model.print()

In [ ]:
# Accessing the learned coefficients
model.coefficients()

In [ ]:
# We can use that to make the coefficient heatmap:
xi_df = pd.DataFrame(model.coefficients().T, index=model.get_feature_names(), columns=['x_dot', 'y_dot'])
xi_df['x_dot_true'] = [0, params_dict['r_x'], 0, -params_dict['c_xx'], -params_dict['c_yx'], 0]
xi_df['y_dot_true'] = [0, 0, params_dict['r_y'], 0, -params_dict['c_xy'], -params_dict['c_yy']]
xi_df['x_dot_error'] = np.abs(xi_df['x_dot'] - xi_df['x_dot_true']) / (np.abs(xi_df['x_dot_true']) + 1e-12)
xi_df['y_dot_error'] = np.abs(xi_df['y_dot'] - xi_df['y_dot_true']) / (np.abs(xi_df['y_dot_true']) + 1e-12)
xi_df['x_dot_error'] = xi_df['x_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['y_dot_error'] = xi_df['y_dot_error'].clip(lower=0, upper=10) # Clip for visualization purposes
xi_df['x_dot_error'] = xi_df['x_dot_error'] * 100  # Convert to percentage
xi_df['y_dot_error'] = xi_df['y_dot_error'] * 100  # Convert to percentage

# Make a heatmap to visualise the error
plt.figure(figsize=(10, 6))
sns.heatmap(xi_df[['x_dot_error', 'y_dot_error']],
            annot=True, fmt=".2f", cmap="viridis", cbar_kws={'label': 'Relative Error'},
            xticklabels=['x_dot', 'y_dot'], yticklabels=model.get_feature_names())
plt.title('Relative Error in Learned Coefficients (in %)');

In [ ]:
xi_df

For more information and tutorials on using `pysindy`, see the [pysindy documentation](https://pysindy.readthedocs.io/en/latest/examples/index.html#id3).

## Model selection

While SINDy provides a framework to nominate potential candidate models, as with any modelling technique we need to decide on the balance between model accuracy and complexity to be able to pick our final model. To do this, we need to tune the sparsity parameter, $\lambda$, in the STLSQ algorithm. This parameter determines how strictly we are enforcing sparsity in the coefficient matrix, and balances the trade-off between model complexity and accuracy of the final model.

Firstly, let us examine the effect of varying $\lambda$ on the inferred model:

In [ ]:
lambda_list = [1e-2, 5e-2, 7.5e-2, 1e-1]
n_vals = len(lambda_list)
_, ax_list = plt.subplots(1, n_vals, figsize=(12, 4), sharey=True, sharex=True)
model_list = []
for i, l in enumerate(lambda_list): # loop over a range of threshold values
    # Perform sparse regression with the current threshold value
    optimiser = ps.STLSQ(threshold=l, alpha=0)
    model = ps.SINDy(optimizer = optimiser, 
                     differentiation_method = differentiation_method,
                     feature_library=feature_library)
    model.fit(observations_df[["x", "y"]].values, t=observations_df["t"].values, feature_names=["x", "y"])
    model_list.append(model) # store for later use

    # Simulate the model
    x_predicted_sindy = model.simulate(x0 = x0,  # initial conditions
                t = t_eval # time points at which to evaluate the solution
            )

    # Re-format into a DataFrame for plotting
    prediction_df_pysindy = pd.DataFrame(x_predicted_sindy, columns=['x', 'y'], index=t_eval)
    prediction_df_pysindy.reset_index(inplace=True)
    prediction_df_pysindy.rename(columns={'index': 't'}, inplace=True)

    # Plot the predicted dynamics on top of the observed data
    ax = ax_list[i]
    sns.scatterplot(data=observations_df, ax=ax, x='t', y='x', label='Species X', 
                    color=LV_PALETTE['x'], s=50, marker='o',
                    edgecolor='black', linewidth=1, zorder=5)
    sns.scatterplot(data=observations_df, ax=ax, x='t', y='y', label='Species Y', 
                    color=LV_PALETTE['y'], s=50, marker='s',
                    edgecolor='black', linewidth=1, zorder=5)
    sns.lineplot(data=prediction_df_pysindy, ax=ax, x='t', y='x', label='SINDy Prediction for X', 
                color=LV_PALETTE['x_sindy'], linewidth=2, zorder=10)
    sns.lineplot(data=prediction_df_pysindy, ax=ax, x='t', y='y', label='SINDy Prediction for Y', 
                color=LV_PALETTE['y_sindy'], linewidth=2, zorder=10)
    ax.legend().remove()
    ax.set_title(rf'$\lambda$ = {l:.1e}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Population Density')
plt.tight_layout()

Let's examine the associated model structures:

In [ ]:
for model in model_list:
    print(f"Threshold: {model.optimizer.threshold}")
    model.print()

As expected, as we increase the threshold $\lambda$, the inferred model becomes sparser, with fewer non-zero coefficients in the coefficient matrix. In parallel, the accuracy of the model becomes lower. Which value of $\lambda$ should we choose?

### Information Criterion based Model Selection

If we view the model obtained for each value of $\lambda$ as an independent model, then this is the classic problem of model selection. As for other statistical model selection problems, a good starting point is to use an information criterion, such as the Akaike Information Criterion (AIC) or the Bayesian Information Criterion (BIC). These criteria take into account both fit and model complexity and provide a principled way to select the best model from a set of candidate models.

Here is how we can compute the AIC for each model:
AIC = 2k - 2ln(L)

where k is the number of parameters in the model, and L is the likelihood of the model given the data. If we assume that the errors are normally distributed (which we implicitely do in using least squares regression), then the log-likelihood can be computed as:

AIC = 2k + n ln(RSS/n)

where RSS is the residual sum of squares, which measures the discrepancy between the observed data and the model predictions.

Importantly, the AIC assumes that the data are independent and identically distributed (i.i.d.), which is not the case for densely sampled time series data, such as ours. Two approaches to deal with this are:
1) Downsample the data to reduce the correlation between observations. 
2) Segment the data into independent windows and compute the AIC for each window independently, and then average the AIC values across windows. This is the approach we will take here (motivated by [Mangan et al 2017](https://royalsocietypublishing.org/doi/full/10.1098/rspa.2017.0009) and adapted from [Urban Fasel's I-X tutorial](https://github.com/urban-fasel/I-X_workshop_2025/blob/main/4_SINDy_model_selection.ipynb)).

In [ ]:
# Prediction error parameters: randomly select nm initial conditions from time series data
n_segments = 10 # number of segments to sample from the time series data
segment_length = 5 # duration of each segment in time units
tspan_segment = np.arange(0, segment_length, sampling_interval) # time span for each segment
n_obs_per_segment = len(tspan_segment) # time steps in each segment
n_obs_total = observations_df.shape[0] # total number of observations in the time series

# Generate the segments
np.random.seed(42)
# Pick segment starting points by randomly picking indices from the time series data, ensuring that we don't
# exceed the bounds of the time series when selecting segments
x0_indices = np.random.choice(n_obs_total - n_obs_per_segment, n_segments, replace=False) 
x0_values = observations_df.loc[x0_indices, ["x", "y"]]

# Plot the segments on top of the original time series data to inspect what they look like
plt.figure(figsize=(12, 6))
sns.scatterplot(data=observations_df, x='t', y='x', label='Species X', 
                color=LV_PALETTE['x'], s=50, marker='o',
                edgecolor='black', linewidth=1, zorder=5)
sns.scatterplot(data=observations_df, x='t', y='y', label='Species Y', 
                color=LV_PALETTE['y'], s=50, marker='s',
                edgecolor='black', linewidth=1, zorder=5)
for i, (idx, row) in enumerate(x0_values.iterrows()):
    t_segment = tspan_segment + observations_df.loc[idx, 't']
    x_segment = observations_segment = observations_df.iloc[idx:idx+n_obs_per_segment]["x"].values
    y_segment = observations_segment = observations_df.iloc[idx:idx+n_obs_per_segment]["y"].values
    curr_colour = sns.color_palette("tab10", n_colors=n_segments)[i]
    plt.plot(t_segment, x_segment, color=curr_colour, linestyle='-', linewidth=10, zorder=10)
    plt.plot(t_segment, y_segment, color=curr_colour, linestyle='-', linewidth=10, zorder=10)

In [ ]:
lambda_list = np.logspace(-7, -1, 10) # range of threshold values to test
model_stats_list = []
for i, l in enumerate(lambda_list): # loop over a range of threshold values
    # Perform sparse regression with the current threshold value
    optimiser = ps.STLSQ(threshold=l, alpha=0)
    model = ps.SINDy(optimizer = optimiser, 
                     differentiation_method = differentiation_method,
                     feature_library=feature_library)
    model.fit(observations_df[["x", "y"]].values, t=observations_df["t"].values, feature_names=["x", "y"])

    # Compute the AIC for each segment
    mse_list = []
    for j, (idx, row) in enumerate(x0_values.iterrows()):
        x0 = row.values # initial conditions for the current segment
        t_segment = tspan_segment + observations_df.loc[idx, 't'] # time span for the current segment
        observations_segment = observations_segment = observations_df.iloc[idx:idx+n_obs_per_segment][["x", "y"]].values # observed data for the current segment
        
        # Simulate the model for the current segment
        model_predictions = model.simulate(x0=x0, t=t_segment)
        
        # Compute the mean squared error (MSE) between the observed data and the model predictions
        # for this segment. We will treat this as an "independent" observation for the purpose of computing the AIC.
        mse = np.mean((observations_segment - model_predictions) ** 2)
        mse_list.append(mse)

    # Compute the AIC
    k = np.count_nonzero(model.coefficients()) # Count the number of non-zero coefficients
    log_L = -n_segments * np.log(np.mean(mse_list)) # log likelihood
    aic = -2 * log_L + 2 * k
    model_stats_list.append({
        "threshold": l,
        "mse": np.mean(mse_list),
        "neg_logL": -log_L,
        "n_nonzero_coeffs": k,
        "aic": aic
    })
# Convert the list of dictionaries into a pandas DataFrame for easier analysis
model_stats_df = pd.DataFrame(model_stats_list)
model_stats_df

In [ ]:
# We can visualize the AIC values by plotting them against the threshold values:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot the final scores, prediction error, and 2k curves
sns.lineplot(data=model_stats_df, x='threshold', y='aic', label='AIC', marker='o')
sns.lineplot(data=model_stats_df, x='threshold', y='neg_logL', label='Negative Log-Likelihood', marker='o')
ax2 = plt.twinx()
sns.lineplot(data=model_stats_df, x='threshold', y='n_nonzero_coeffs', label='Number of Non-Zero Coefficients', marker='o', ax=ax2, color='green')
ax.set_xscale('log')

### Stability Selection

A further interesting technique is so-called stability selection, which was introduced by [Maddu et al 2022](https://royalsocietypublishing.org/doi/full/10.1098/rspa.2021.0916). This technique involves repeatedly applying the SINDy algorithm to different subsets of the data and then selecting the features that are consistently selected across these subsets. The idea is that if a feature is truly relevant to the underlying dynamics, it should be selected consistently across different subsets of the data, while irrelevant features will be selected less consistently. This approach can help provide a sense of the uncertainty quantification for the inferred model structure, and can be used to guide the choice of the sparsity parameter $\lambda$.

This code has been adapted from [Urban Fasel's I-X tutorial](https://github.com/urban-fasel/I-X_workshop_2025/blob/main/4_SINDy_model_selection.ipynb).

In [ ]:
n_subsamples = 50          # number of bootstrap subsamples (Maddu et al. recommend 250)
subsample_size = observations_df.shape[0] // 2  # use half the data per subsample
lambda_list = np.logspace(-7, -1, 10) # range of threshold values to test

# Get number of library features
feature_library.fit(observations_df[["x", "y"]].values)
n_library_terms = feature_library.n_output_features_
n_state_vars = 2  # x and y

# Arrays to store coefficients and active terms across subsamples and thresholds
n_thresholds = len(lambda_list)
coeffs = np.zeros((n_state_vars, n_library_terms, n_thresholds, n_subsamples))
coeffs_active = np.zeros((n_state_vars, n_library_terms, n_thresholds, n_subsamples), dtype=bool)

# Compute derivatives once on the full dataset
X = observations_df[["x", "y"]].values
X_dot = differentiation_method._differentiate(X, observations_df["t"].values)

# Sweep over subsamples and thresholds
np.random.seed(42)
for j in range(n_subsamples):
    # Randomly sample half the observations (without replacement)
    indices = np.random.choice(observations_df.shape[0], subsample_size, replace=False)
    indices = np.sort(indices)  # Sort indices to maintain temporal order

    for i, l in enumerate(lambda_list):
        optimiser = ps.STLSQ(threshold=l)
        model = ps.SINDy(
            optimizer=optimiser,
            differentiation_method=differentiation_method,
            feature_library=feature_library
        )
        model.fit(X[indices, :], t=observations_df["t"].values[indices],
                  x_dot=X_dot[indices, :], feature_names=["x", "y"])

        coeffs[:, :, i, j] = model.coefficients()
        coeffs_active[:, :, i, j] = (model.coefficients() != 0)

In [ ]:
# Plot the results
stability_threshold = 0.8 # minimum inclusion probability to consider a term "stable"
state_names = ["x", "y"]
feature_names = feature_library.get_feature_names_out(input_features=["x", "y"])

# Calculate the mean inclusion probability over the subsamples
mean_inclusion_prob = np.mean(coeffs_active, axis=3)  # shape: (n_state_vars, n_library_terms, n_thresholds)

fig, axs = plt.subplots(1, n_state_vars, figsize=(14, 4))

for i, (ax, state) in enumerate(zip(axs, state_names)):
    # mean_inclusion_prob[i] has shape (n_library_terms, n_thresholds)
    # rows = library terms, columns = lambda values
    sns.heatmap(
        mean_inclusion_prob[i],
        ax=ax,
        vmin=0, vmax=1,
        cmap='YlOrRd',
        xticklabels=[f"{l:.1e}" for l in lambda_list],
        yticklabels=feature_names,
        cbar_kws={'label': 'Inclusion probability'},
        mask=mean_inclusion_prob[i] < stability_threshold, # Mask terms below our stability threshold
    )
    ax.set_facecolor('lightgray')  # Set background color for masked areas
    ax.set_title(rf'$\dot{{{state}}}$')
    ax.set_xlabel(r'Regularization threshold ($\lambda$)')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

axs[0].set_ylabel('Library term')
plt.tight_layout()

Fundamentally, the "right" approach to model selection depends on the goal of the analysis, and you should choose the approach that best aligns with your goals!

## Extensions
- SINDy with control
- Noisy/sparse data
- SINDy for PDEs
- Weak form SINDy
- SINDy for model-predictive control
- Bayesian SINDy
- Unobserved state variables

## Summary

**Key takeaways from this tutorial:**
- SINDy is a powerful and flexible method for discovering governing equations from data.
- The key idea is to recast the problem of discovering governing equations as a sparse, linear regression problem, by proposing a library of candidate functions and estimating the coefficients that combine these functions to form the governing equations.
- This regression happens on the derivatives of the state variables. These can be estimated from the data, or may be observed explicitly in some cases (e.g. via accelerometers, gyroscopes, etc).

**Words of caution:**

To conclude, I want to highlight three important points about the assumptions which SINDy is built on, and the implications of these assumptions for the use of SINDy in practice:
- **SINDy's use of derivatives**:
    1. SINDy does not require forward simulation during the inference process, which makes it computationally efficient and scalable to large datasets. 
    2. But, by virtue of not performing forward simulation, there is no immediate guarantee that the model generated by SINDy will be integrable or generate physical solutions, in particular with large and highly correlated libraries. This is a key reason why model selection and validation is so important when using SINDy in practice.
    3. Garbage in = Garbage out. The quality of the inferred model is highly dependent on the quality of the derivative data, and derivatives in particular are sensitive to noise in the data. There are smoothing approaches and weak forms of SINDy that can help with this (see xxx).
- **SINDy's use of a library of candidate functions**: Unfortunately, SINDy is not a magic bullet, and it is still advicable to be conscientous about the choice of library and the implications of that choice for the inferred model. 
- **SINDy assumes that the governing equations are linear in their coefficients**: This is a key assumption of SINDy, and it is important because many well-known models in physics and biology are not linear in their coefficients. For example, the canonical form of Lotka-Volterra model in which we specify the population's carryng capacity is actually not linear:
\begin{align}
    \dot{x} &= r_x \left(1 - \frac{x + c_{yx} y}{K_x}\right) x
             = r_x x - \frac{r_x}{K_x} x^2 - \frac{r_x c_{yx}}{K_x} xy \\[1ex]
    \dot{y} &= r_y \left(1 - \frac{y + c_{xy} x}{K_y}\right) y
             = r_y y - \frac{r_y}{K_y} y^2 - \frac{r_y c_{xy}}{K_y} xy
\end{align}

That being said, this can often be fixed through re-parameterisation (as done here). For a full discussion of how to deal with rational functions see [Mangan et al 2016](https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7809160).